# MNIST: multilayer perceptrons with PyTorch

Multilayer perceptrons (MLPs) can be used for both regression and classification. In this notebook we train a classic MLP to recognize handwritten digits from the MNIST dataset, compare it with a model that uses dropout, and investigate the resulting classifiers.

## Learning objectives

After working through this notebook, you should be able to:

- prepare a built-in vision dataset with transformations and data loaders;
- explain why a classification model returns logits rather than probabilities during training;
- implement and train an MLP with and without dropout;
- use validation evidence to select a model without consulting the test set;
- interpret a confusion matrix and investigate sensitivity to initialization.


## Configuration

The complete experiment executes 100 epochs, including ten repeated training runs. That is computationally expensive. The default quick configuration preserves the workflow with shorter runs; set `FULL_EXPERIMENT = True` for the complete experiment.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import MNIST
from torchvision.transforms import v2

In [ ]:
FULL_EXPERIMENT = False

BATCH_SIZE = 128
EPOCHS = 100 if FULL_EXPERIMENT else 5
SENSITIVITY_RUNS = 10 if FULL_EXPERIMENT else 3
SENSITIVITY_EPOCHS = 100 if FULL_EXPERIMENT else 5

SPLIT_SEED = 1234
MODEL_SEED = 4958
DATA_ORDER_SEED = 2718

DATA_DIR = Path.cwd() / "data"
MODEL_DIR = Path.cwd() / "models"

## Reproducibility and device selection

PyTorch uses its own random-number generator. A fixed seed makes the CPU run reproducible under the same software and hardware configuration. Exact results can still differ across devices or PyTorch versions because some accelerator operations are nondeterministic or use different numerical implementations.


In [ ]:
def seed_everything(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(MODEL_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Data preparation

### Obtaining and transforming the dataset

MNIST is provided by `torchvision.datasets`.A PyTorch `Dataset` retrieves individual image-label pairs. A `DataLoader` will later handle batching and shuffling.

The transformation converts each image to a `float32` tensor and scales its pixel values from the integer range 0–255 to the floating-point range 0–1. Flattening is part of the model, so the dataset retains the meaningful image shape `[channel, height, width]`.

In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])


In [ ]:
full_train_dataset = MNIST(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform,
)

test_dataset = MNIST(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform,
)


### Validation set

We split the 60,000 original training examples into 45,000 training and 15,000 validation examples. The test set remains untouched while models and training choices are being compared.


In [ ]:
split_generator = torch.Generator().manual_seed(SPLIT_SEED)

train_dataset, validation_dataset = random_split(
    full_train_dataset,
    [45_000, 15_000],
    generator=split_generator,
)

len(train_dataset), len(validation_dataset), len(test_dataset)


### Data loaders

The training loader shuffles examples at every epoch. Supplying its generator explicitly lets us reproduce the same sequence of shuffles when comparing models. Validation and test loaders do not shuffle because example order does not affect their metrics. When CUDA is used, pinned CPU memory and non-blocking device transfers can work together to make host-to-device copies asynchronous.


In [ ]:
using_cuda = device.type == "cuda"


def make_train_loader(seed=DATA_ORDER_SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=using_cuda,
    )


validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=using_cuda,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=using_cuda,
)


### Verification

Before training, inspect an image, its tensor properties, and its integer class label.


In [ ]:
image, label = train_dataset[0]

print(f"shape: {tuple(image.shape)}")
print(f"dtype: {image.dtype}")
print(f"value range: [{image.min().item():.1f}, {image.max().item():.1f}]")
print(f"label: {label}")

plt.imshow(image.squeeze(0), cmap="gray")
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()


The target is an integer class index rather than a one-hot vector. This is exactly what `nn.CrossEntropyLoss` expects.

## Classic multilayer perceptron

The network has 784 inputs (28 × 28 pixels), two hidden layers with 512 units and ReLU activations, and 10 output values—one per digit. `nn.Flatten` converts each image to 784 features inside the model.

The final layer returns **logits**, not Softmax probabilities. `nn.CrossEntropyLoss` combines the numerically stable LogSoftmax and negative log-likelihood operations internally.


In [ ]:
def make_mlp(dropout_probability=0.0):
    """Build the 784 -> 512 -> 10 classifier described above."""
    # TODO: Include flattening, a ReLU hidden layer, optional dropout, and
    # unnormalized output logits suitable for CrossEntropyLoss.
    raise NotImplementedError


In [ ]:
seed_everything(MODEL_SEED)
model = make_mlp().to(device)

model


In [ ]:
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Trainable parameters: {parameter_count:,}")


### Loss function and optimizer

The loss measures multiclass classification error, while stochastic gradient descent updates the weights. The learning rate `lr` is 0.01.

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)


### Training and evaluation functions

The PyTorch training loop has the following steps:

1. switch the model to training mode;
2. compute logits and loss for a batch;
3. compute gradients with backpropagation;
4. update parameters with the optimizer;
5. aggregate loss and accuracy over the epoch.

Evaluation switches to evaluation mode—important for dropout—and disables gradient tracking.


In [ ]:
def train_one_epoch(
    model,
    data_loader,
    loss_function,
    optimizer,
    device,
    non_blocking=False,
):
    """Train for one epoch and return example-weighted loss and accuracy."""
    # TODO: Switch to training mode and, for each batch, move the data to the
    # device, clear gradients, compute logits and loss, backpropagate, update
    # parameters, and accumulate metrics over individual examples.
    raise NotImplementedError


In [ ]:
@torch.inference_mode()
def evaluate(
    model,
    data_loader,
    loss_function,
    device,
    non_blocking=False,
):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for features, targets in data_loader:
        features = features.to(device, non_blocking=non_blocking)
        targets = targets.to(device, non_blocking=non_blocking)

        logits = model(features)
        loss = loss_function(logits, targets)

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == targets).sum().item()
        total_examples += batch_size

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


In [ ]:
def fit(
    model,
    train_loader,
    validation_loader,
    loss_function,
    optimizer,
    epochs,
    device,
    non_blocking=False,
    verbose=True,
):
    history = {
        "loss": [],
        "accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(
            model,
            train_loader,
            loss_function,
            optimizer,
            device,
            non_blocking,
        )
        validation_metrics = evaluate(
            model,
            validation_loader,
            loss_function,
            device,
            non_blocking,
        )

        history["loss"].append(train_metrics["loss"])
        history["accuracy"].append(train_metrics["accuracy"])
        history["val_loss"].append(validation_metrics["loss"])
        history["val_accuracy"].append(validation_metrics["accuracy"])

        if verbose:
            print(
                f"Epoch {epoch:3d}/{epochs}: "
                f"loss={train_metrics['loss']:.4f}, "
                f"accuracy={train_metrics['accuracy']:.4f}, "
                f"val_loss={validation_metrics['loss']:.4f}, "
                f"val_accuracy={validation_metrics['accuracy']:.4f}"
            )

    return history


### Training


In [ ]:
train_loader = make_train_loader()

model_history = fit(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    loss_function=loss_function,
    optimizer=optimizer,
    epochs=EPOCHS,
    device=device,
    non_blocking=using_cuda,
)


Plot the history of the training process. Comparing training and validation curves helps diagnose underfitting and overfitting.


In [ ]:
def plot_history(history):
    epochs = np.arange(1, len(history["loss"]) + 1)

    figure, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["loss"], label="Training")
    axes[0].plot(epochs, history["val_loss"], label="Validation")
    axes[0].set(xlabel="Epoch", ylabel="Loss")
    axes[0].legend()

    axes[1].plot(epochs, history["accuracy"], label="Training")
    axes[1].plot(epochs, history["val_accuracy"], label="Validation")
    axes[1].set(xlabel="Epoch", ylabel="Accuracy")
    axes[1].legend(loc="lower right")

    figure.tight_layout()


plot_history(model_history)


Compare performance on the training and validation sets. These are the development results used to diagnose and select models. The test set remains untouched until every model choice has been frozen.


In [ ]:
def evaluate_development_sets(
    model,
    device,
    pin_memory=False,
    non_blocking=False,
):
    loaders = {
        "train": DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            pin_memory=pin_memory,
        ),
        "validation": validation_loader,
    }

    results = {}
    for name, data_loader in loaders.items():
        results[name] = evaluate(
            model,
            data_loader,
            loss_function,
            device,
            non_blocking,
        )
        metrics = results[name]
        print(
            f"{name:10s}: loss={metrics['loss']:.4f}, "
            f"accuracy={metrics['accuracy']:.4f}"
        )

    return results


model_development_results = evaluate_development_sets(
    model,
    device,
    pin_memory=using_cuda,
    non_blocking=using_cuda,
)


### Save and reload the model

Saving a `state_dict` stores learned parameters without serializing arbitrary Python objects. To reload it, recreate the architecture and then load the weights.


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / "mnist_mlp.pt"

torch.save(model.state_dict(), model_path)
print(f"Saved weights to {model_path}")


In [ ]:
reloaded_model = make_mlp().to(device)
reloaded_model.load_state_dict(
    torch.load(model_path, map_location=device, weights_only=True)
)

reloaded_validation_metrics = evaluate(
    reloaded_model,
    validation_loader,
    loss_function,
    device,
    non_blocking=using_cuda,
)
reloaded_validation_metrics


## Dropout layers

Dropout randomly sets some activations to zero during training, discouraging the network from relying too strongly on particular paths. It is disabled automatically in evaluation mode.

Before running the cell, predict how dropout will affect:

- training loss and accuracy;
- the gap between training and validation performance;
- test performance.

We reset both the model seed and data-loader seed so that the comparison changes the architecture while keeping initialization and batch order as comparable as possible.


In [ ]:
seed_everything(MODEL_SEED)
dropout_model = make_mlp(dropout_probability=0.2).to(device)
dropout_optimizer = torch.optim.SGD(dropout_model.parameters(), lr=0.01)
dropout_train_loader = make_train_loader(DATA_ORDER_SEED)

dropout_model


In [ ]:
dropout_model_history = fit(
    model=dropout_model,
    train_loader=dropout_train_loader,
    validation_loader=validation_loader,
    loss_function=loss_function,
    optimizer=dropout_optimizer,
    epochs=EPOCHS,
    device=device,
    non_blocking=using_cuda,
)


In [ ]:
plot_history(dropout_model_history)
dropout_model_development_results = evaluate_development_sets(
    dropout_model,
    device,
    pin_memory=using_cuda,
    non_blocking=using_cuda,
)


In [ ]:
dropout_model_path = MODEL_DIR / "mnist_mlp_dropout.pt"
torch.save(dropout_model.state_dict(), dropout_model_path)
print(f"Saved weights to {dropout_model_path}")


## Understanding and selecting the model

The selection rule must be declared before consulting the test set. Here we select the candidate with the lowest validation loss. Accuracy remains useful to report, but loss uses the full predicted distribution and is usually more sensitive when choosing between classifiers.

All diagnostics below use validation data. If a diagnostic motivates another modelling change, retrain and repeat the validation analysis before proceeding to the final test.


In [ ]:
candidate_models = {
    "classic": model,
    "dropout": dropout_model,
}
candidate_results = {
    "classic": model_development_results,
    "dropout": dropout_model_development_results,
}

# TODO: Select the candidate with the lowest validation loss. Do not use
# the test set for this choice.
selected_model_name = ...
selected_model = ...
analysis_model = selected_model

print(
    f"Selected {selected_model_name!r} model: "
    f"validation loss="
    f"{candidate_results[selected_model_name]['validation']['loss']:.4f}"
)


### Validation confusion matrix

A confusion matrix reveals which classes the model confuses. Rows represent true labels and columns represent predicted labels. We compute it on the validation set so it can support diagnosis without exposing the test set.


In [ ]:
@torch.inference_mode()
def predict_classes(
    model, data_loader, device, non_blocking=False
):
    model.eval()
    predicted_batches = []
    target_batches = []

    for features, targets in data_loader:
        features = features.to(device, non_blocking=non_blocking)
        logits = model(features)
        predicted_batches.append(logits.argmax(dim=1).cpu())
        target_batches.append(targets)

    return torch.cat(target_batches), torch.cat(predicted_batches)


validation_targets, validation_predictions = predict_classes(
    analysis_model,
    validation_loader,
    device,
    non_blocking=using_cuda,
)

confusion_matrix = torch.bincount(
    10 * validation_targets + validation_predictions,
    minlength=10 * 10,
).reshape(10, 10)

confusion_matrix


In [ ]:
def plot_confusion_matrix(matrix, normalize=False, cmap="Blues"):
    values = matrix.detach().cpu().numpy().astype(np.float64)

    if normalize:
        row_totals = values.sum(axis=1, keepdims=True)
        display_values = np.divide(
            values,
            row_totals,
            out=np.zeros_like(values),
            where=row_totals != 0,
        )
        color_values = display_values
        value_format = ".3f"
    else:
        display_values = values
        color_values = np.log1p(values)
        value_format = ".0f"

    figure, axes = plt.subplots(figsize=(7, 6))
    image = axes.imshow(color_values, interpolation="nearest", cmap=cmap)
    figure.colorbar(image, ax=axes)

    classes = np.arange(10)
    axes.set(
        xticks=classes,
        yticks=classes,
        xlabel="Predicted label",
        ylabel="True label",
    )

    threshold = color_values.max() / 2.0
    for row in range(10):
        for column in range(10):
            axes.text(
                column,
                row,
                format(display_values[row, column], value_format),
                ha="center",
                va="center",
                color=(
                    "white"
                    if color_values[row, column] > threshold
                    else "black"
                ),
                fontsize=8,
            )

    figure.tight_layout()


plot_confusion_matrix(confusion_matrix, normalize=True)


Inspect the largest off-diagonal entries. Are the most common confusions visually plausible? A confusion matrix describes error patterns, but it does not by itself explain their cause. Inspect validation images before proposing explanations.

### Sensitivity to initial conditions

Training is stochastic. To isolate sensitivity to parameter initialization, every repeated run below uses:

- a different model seed;
- the same train/validation split;
- the same sequence of shuffled mini-batches.

This controls the data order rather than silently changing initialization and sampling together. The test set is deliberately excluded from this exploratory analysis.


In [ ]:
dataset_loaders = {
    "train": DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=using_cuda,
    ),
    "validation": validation_loader,
}

losses = {name: [] for name in dataset_loaders}
accuracies = {name: [] for name in dataset_loaders}

for run in range(SENSITIVITY_RUNS):
    run_seed = MODEL_SEED + run
    print(
        f"Training model {run + 1}/{SENSITIVITY_RUNS} "
        f"with seed {run_seed}"
    )

    seed_everything(run_seed)
    repeated_model = make_mlp(dropout_probability=0.2).to(device)
    repeated_optimizer = torch.optim.SGD(
        repeated_model.parameters(), lr=0.01
    )
    repeated_train_loader = make_train_loader(DATA_ORDER_SEED)

    _ = fit(
        model=repeated_model,
        train_loader=repeated_train_loader,
        validation_loader=validation_loader,
        loss_function=loss_function,
        optimizer=repeated_optimizer,
        epochs=SENSITIVITY_EPOCHS,
        device=device,
        non_blocking=using_cuda,
        verbose=False,
    )

    for name, data_loader in dataset_loaders.items():
        metrics = evaluate(
            repeated_model,
            data_loader,
            loss_function,
            device,
            non_blocking=using_cuda,
        )
        losses[name].append(metrics["loss"])
        accuracies[name].append(metrics["accuracy"])


In [ ]:
def plot_run_variation(losses, accuracies):
    names = list(losses)
    positions = np.arange(len(names))
    jitter = np.linspace(-0.08, 0.08, SENSITIVITY_RUNS)

    figure, axes = plt.subplots(1, 2, figsize=(12, 4))

    for position, name in zip(positions, names):
        axes[0].scatter(
            position + jitter, losses[name], label=name
        )
        axes[1].scatter(
            position + jitter, accuracies[name], label=name
        )

    for axis, ylabel in zip(axes, ["Loss", "Accuracy"]):
        axis.set_xticks(positions, names)
        axis.set_ylabel(ylabel)

    figure.tight_layout()


plot_run_variation(losses, accuracies)


In [ ]:
for name in dataset_loaders:
    loss_values = np.asarray(losses[name])
    accuracy_values = np.asarray(accuracies[name])
    print(
        f"{name:10s}: "
        f"loss={loss_values.mean():.4f} ± {loss_values.std(ddof=0):.4f}, "
        f"accuracy={accuracy_values.mean():.4f} "
        f"± {accuracy_values.std(ddof=0):.4f}"
    )


## Interpretation questions

Answer these questions using only the training and validation evidence:

1. Does dropout reduce the training–validation gap? Does that necessarily mean it is the better model?
2. Which digit pairs have the largest off-diagonal validation confusion counts? Inspect several corresponding images before proposing an explanation.
3. Is variation across initialization seeds large enough to change your model-selection conclusion?
4. Would repeating the experiment with different train/validation splits answer the same scientific question? Why or why not?
5. Based on the declared validation-loss criterion, which model was selected? Would selecting by validation accuracy change the choice?
6. Which information—software versions, device, seeds, split, transformations, and hyperparameters—would someone need to reproduce the comparison?


## Final test evaluation

The model architecture and selection rule are now frozen. Evaluate the selected model on the test set exactly once to estimate its performance on unseen data.

Do not use this result to choose another model or tune another hyperparameter: doing so would turn the test set into another validation set. Further development requires returning to the validation workflow and reserving a fresh test or audit set for a new final evaluation.


In [ ]:
selected_model_path = MODEL_DIR / "mnist_mlp_selected.pt"
torch.save(selected_model.state_dict(), selected_model_path)

final_test_metrics = evaluate(
    selected_model,
    test_loader,
    loss_function,
    device,
    non_blocking=using_cuda,
)

print(f"Selected model: {selected_model_name}")
print(
    f"Final test: loss={final_test_metrics['loss']:.4f}, "
    f"accuracy={final_test_metrics['accuracy']:.4f}"
)
print(f"Saved selected weights to {selected_model_path}")
